# CNN hyperparameter optimization

Optuna searches the **vision policy's architecture** — CNN channels, kernel
sizes, activation, MLP head width — together with the usual training
hyperparameters (lr, entropy). Trials run **in this process**, one after
another (Genesis scenes rebuild fine in-process); the trainer's periodic
deterministic evals stream to a Hyperband pruner, so bad architectures die
after their first eval instead of wasting the full budget.

Needs the Madrona renderer (local NVIDIA machine — not Colab).
The study persists in sqlite: re-run the notebook to add trials.

In [ ]:
import optuna
from deepracer_genesis.experiment import (
    AsymmetricCameraPolicy,
    CameraEnvironment,
    DomainRandomizationCamera,
    DomainRandomizationTrackAppearance,
    run,
)

N_TRIALS = 12
STEPS = 4_000_000          # per trial (~6 min each at ~12k steps/s)
EVAL_EVERY = 500_000       # pruning signal cadence
NUM_ENVS = 128
METRIC = "mean_progress_m" # completion_rate saturates late; progress ranks earlier

## Search space

Strides are fixed at (4, 2, 1) so every sampled architecture stays valid at
160x120; kernels/channels/activation/head vary. Add anything else the spec
exposes (lr, entropy, DR strength, resolution...) the same way.

In [ ]:
def sample_spec(trial: optuna.Trial):
    cnn = {
        "channels": [trial.suggest_categorical("ch1", [16, 32]),
                     trial.suggest_categorical("ch2", [32, 64]),
                     trial.suggest_categorical("ch3", [64, 128])],
        "kernels": [trial.suggest_categorical("k1", [5, 8]),
                    trial.suggest_categorical("k2", [3, 5]),
                    3],
        "strides": [4, 2, 1],
        "activation": trial.suggest_categorical("act", ["relu", "elu"]),
    }
    mlp = {"hidden": trial.suggest_categorical("hidden",
                                               [(256, 128), (512, 256)])}
    spec = (CameraEnvironment(resolution=(160, 120), num_envs=NUM_ENVS)
            >> DomainRandomizationTrackAppearance(strength=0.6)
            >> DomainRandomizationCamera(brightness=(0.7, 1.3), hue=0.05)
            >> AsymmetricCameraPolicy(cnn=cnn, mlp=mlp)
            ).build(ablation_group="cnn_hpo",
                    total_env_steps=STEPS, eval_every_steps=EVAL_EVERY)
    from deepracer_genesis.experiment.ablation import override
    spec = override(spec, "algorithm.ppo.lr",
                    trial.suggest_float("lr", 1e-4, 3e-3, log=True))
    spec = override(spec, "algorithm.ppo.entropy_coef",
                    trial.suggest_float("entropy_coef", 3e-3, 3e-2, log=True))
    return spec


def objective(trial: optuna.Trial) -> float:
    spec = sample_spec(trial)

    def report(frames, metrics):
        trial.report(metrics[METRIC], frames)
        if trial.should_prune():
            raise optuna.TrialPruned()

    record = run(spec, root="runs/cnn_hpo", force=True, on_eval=report)
    return float(record.metrics[METRIC])

In [ ]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=0),
    pruner=optuna.pruners.HyperbandPruner(min_resource=EVAL_EVERY,
                                          max_resource=STEPS,
                                          reduction_factor=3),
    study_name="cnn_hpo",
    storage="sqlite:///runs/cnn_hpo/study.db",
    load_if_exists=True,
)
study.optimize(objective, n_trials=N_TRIALS)
print("best:", study.best_value, study.best_params)

## What mattered?

In [ ]:
import optuna.visualization.matplotlib as vis
vis.plot_param_importances(study);

In [ ]:
vis.plot_optimization_history(study);

## Ship the winner

Re-run the best config longer, watch it drive, export it:

In [ ]:
best = optuna.trial.FixedTrial(study.best_params)
spec = sample_spec(best)
record = run(spec, root="runs/cnn_hpo", total_env_steps=20_000_000)
print(record.metrics)

from deepracer_genesis.experiment.visualize import rollout_video
rollout_video(spec, root="runs/cnn_hpo")

from deepracer_genesis.experiment.export import export_policy
export_policy(spec, root="runs/cnn_hpo")     # policy.onnx + model_card.json